# Clase 091 — La maldición de la dimensionalidad

Por qué los algoritmos basados en distancia (kNN, k-means, SVM-RBF) degradan al crecer el número de features: el espacio se vuelve mayormente vacío, las distancias colapsan a un mismo valor y aparece la *manifold hypothesis*.

Requiere: `numpy`, `scikit-learn`, `matplotlib`.

## 1. Volumen del borde del hipercubo

En `[0,1]^d`, la fracción de volumen a menos de `0.01` del borde es `1 - 0.98^d`. Crece rápido: casi todo el volumen vive pegado al borde.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

dims = np.arange(1, 201)
frac_borde = 1 - 0.98 ** dims
for d in (2, 10, 100):
    print(f"d={d:>3}: fraccion cerca del borde = {1 - 0.98**d:.4f}")
assert frac_borde[99] > 0.85, "en d=100 casi todo el volumen esta en el borde"

plt.figure(figsize=(7, 4))
plt.plot(dims, frac_borde, color="#37a")
plt.axhline(0.99, ls="--", color="#c33", lw=0.8, label="99%")
plt.xlabel("dimension d"); plt.ylabel("fraccion cerca del borde")
plt.title("El volumen se concentra en el borde")
plt.legend(); plt.tight_layout(); plt.show()

## 2. Concentración de la medida

Sampleamos puntos uniformes y medimos `(d_max - d_min) / d_min` sobre las distancias pairwise. Debe tender a 0: en alta dimensión todas las distancias se igualan.

In [ ]:
from sklearn.metrics import pairwise_distances

def ratio_distancias(d, n=500, seed=42):
    r = np.random.default_rng(seed)
    X = r.random((n, d))
    D = pairwise_distances(X)
    iu = np.triu_indices(n, k=1)
    dd = D[iu]
    return (dd.max() - dd.min()) / dd.min()

ratios = {d: ratio_distancias(d) for d in (2, 10, 100, 1000)}
for d, r in ratios.items():
    print(f"d={d:>4}: (d_max - d_min)/d_min = {r:.4f}")
assert ratios[1000] < ratios[2], "el ratio debe caer al crecer d"
print("\nLa nocion de 'mas cercano' se vuelve ruido en alta dimension.")

## 3. La distancia al vecino más cercano crece con `d`

Con `n` fijo, el 1-NN está cada vez más lejos: el espacio se vacía.

In [ ]:
from sklearn.neighbors import NearestNeighbors

dims = [2, 5, 10, 25, 50, 100, 200]
dist_1nn = []
for d in dims:
    X = rng.random((1000, d))
    nn = NearestNeighbors(n_neighbors=2).fit(X)
    dst, _ = nn.kneighbors(X)
    dist_1nn.append(dst[:, 1].mean())  # col 0 es el propio punto

for d, m in zip(dims, dist_1nn):
    print(f"d={d:>3}: distancia media al 1-NN = {m:.3f}")

plt.figure(figsize=(7, 4))
plt.plot(dims, dist_1nn, "o-", color="#3a7")
plt.xlabel("dimension d"); plt.ylabel("dist. media al 1-NN")
plt.title("El vecino mas cercano se aleja al crecer d")
plt.tight_layout(); plt.show()

## 4. kNN degrada al agregar features de ruido

Partimos de un problema separable y le sumamos dimensiones de ruido puro. El accuracy de kNN debe caer.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

X_base, y = make_classification(n_samples=500, n_features=2, n_informative=2,
                                n_redundant=0, n_clusters_per_class=1,
                                class_sep=1.5, random_state=42)

extra_ruido = [0, 8, 48, 198]  # d total = 2, 10, 50, 200
accs = []
for k_noise in extra_ruido:
    noise = rng.normal(size=(500, k_noise))
    X = np.hstack([X_base, noise])
    acc = cross_val_score(KNeighborsClassifier(n_neighbors=5), X, y, cv=5, n_jobs=1).mean()
    accs.append(acc)
    print(f"d={2 + k_noise:>3}: accuracy kNN = {acc:.4f}")

assert accs[0] > accs[-1], "el accuracy debe caer al inyectar ruido"

plt.figure(figsize=(7, 4))
plt.plot([2 + k for k in extra_ruido], accs, "o-", color="#c33")
plt.xlabel("dimension total d"); plt.ylabel("accuracy CV")
plt.title("kNN se degrada con features irrelevantes")
plt.tight_layout(); plt.show()

## 5. Manifold hypothesis empírica sobre `load_digits`

`digits` tiene 64 features (8x8) pero su dimensión intrínseca es mucho menor: pocos componentes PCA explican casi toda la varianza.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

X_dig = load_digits().data
pca = PCA().fit(X_dig)
cum = np.cumsum(pca.explained_variance_ratio_)

for umbral in (0.90, 0.95, 0.99):
    n = int(np.searchsorted(cum, umbral) + 1)
    print(f"{int(umbral*100)}% de varianza -> {n} componentes (de 64)")

n95 = int(np.searchsorted(cum, 0.95) + 1)
assert n95 <= 30, "digits debe tener dimension intrinseca <= 30 al 95%"

plt.figure(figsize=(7, 4))
plt.plot(np.arange(1, 65), cum, color="#37a")
plt.axhline(0.95, ls="--", color="#c33", lw=0.8, label="95%")
plt.xlabel("n componentes"); plt.ylabel("varianza acumulada")
plt.title("digits (d=64): dimension intrinseca baja")
plt.legend(); plt.tight_layout(); plt.show()

## Ejercicios

1. **Volumen del borde.** Calculá `1 - 0.98^d` para `d = 2, 10, 100` y verificá que en `d=100` casi todo el volumen está pegado al borde.
2. **Concentración.** Repetí la sección 2 con puntos gaussianos en vez de uniformes. ¿El ratio también tiende a 0?
3. **kNN vs árboles.** Repetí la sección 4 con `RandomForestClassifier`. ¿Se degrada tanto como kNN al agregar ruido?
4. **Dimensión intrínseca.** Sobre `load_breast_cancer`, contá cuántos componentes PCA explican 95% de la varianza.

## Conclusiones

- En alta dimensión el espacio se vacía y las distancias se igualan: kNN, k-means y SVM-RBF pierden poder.
- La maldición es sobre las **features** (`d`), no sobre las muestras (`n`); más datos ayuda solo asintóticamente.
- Los árboles y modelos lineales regularizados son mucho más robustos al ruido dimensional.
- La *manifold hypothesis* justifica reducir dimensionalidad: los datos reales viven en un subespacio de dimensión intrínseca baja.

## ✅ Soluciones de los ejercicios

Cinco ejercicios numéricos sobre por qué la alta dimensión rompe la intuición geométrica y degrada a los métodos basados en distancia. `n_jobs=1`.

**Ejercicio 1 — Volumen del borde.** Fracción del hipercubo `[0,1]^d` a menos de 0.01 del borde: `1 - 0.98^d`.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
for d in [2, 10, 100]:
    frac = 1 - 0.98 ** d
    print(f'd={d:4d} -> {frac:.4%} del volumen esta pegado al borde')
assert (1 - 0.98 ** 100) > 0.8, 'en 100D casi todo el volumen esta en la cascara'
print('En alta dimension casi todos los puntos viven cerca del borde.')

**Ejercicio 2 — Concentración de distancias.** `(d_max - d_min)/d_min` sobre distancias pairwise tiende a 0 al crecer `d`.

In [ ]:
from scipy.spatial.distance import pdist
rng = np.random.default_rng(42)
for d in [2, 10, 100, 1000]:
    P = rng.random((1000, d))
    dd = pdist(P)
    ratio = (dd.max() - dd.min()) / dd.min()
    print(f'd={d:5d} -> (dmax-dmin)/dmin = {ratio:.3f}')
print('El contraste entre el vecino mas cercano y el mas lejano se desvanece.')

**Ejercicio 3 — Distancia al 1-NN.** Con `n=1000`, la distancia media al vecino más cercano crece con `d`.

In [ ]:
from sklearn.neighbors import NearestNeighbors
ds = [2, 5, 10, 20, 50, 100, 200]
medias = []
for d in ds:
    P = rng.random((1000, d))
    nn = NearestNeighbors(n_neighbors=2).fit(P)
    dist, _ = nn.kneighbors(P)
    medias.append(dist[:, 1].mean())
plt.figure(figsize=(6, 4))
plt.plot(ds, medias, 'o-'); plt.xlabel('dimension d'); plt.ylabel('dist. media al 1-NN')
plt.title('El "vecino mas cercano" se aleja con d'); plt.tight_layout(); plt.show()
assert medias[-1] > medias[0]
print('media 1-NN:', [round(m, 3) for m in medias])

**Ejercicio 4 — kNN degrada con ruido.** Agregamos features de ruido puro: la accuracy de kNN cae.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
Xc, yc = make_classification(n_samples=500, n_features=2, n_informative=2,
                             n_redundant=0, random_state=42)
accs = []
for d in [2, 10, 50, 200]:
    ruido = rng.normal(size=(500, d - 2))
    Xd = np.hstack([Xc, ruido])
    accs.append(cross_val_score(KNeighborsClassifier(5), Xd, yc, cv=5).mean())
for d, a in zip([2, 10, 50, 200], accs):
    print(f'  d={d:3d} ({d-2} de ruido) -> acc {a:.4f}')
assert accs[0] > accs[-1], 'agregar ruido debe degradar a kNN'
print('kNN se ahoga en dimensiones irrelevantes: las distancias se vuelven ruido.')

**Ejercicio 5 — Dimensión intrínseca (manifold).** Cuántos componentes PCA explican el 95% de la varianza de `load_digits` (64 features originales).

In [ ]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
Xd, _ = load_digits(return_X_y=True)
p = PCA(n_components=0.95).fit(Xd)
print(f'componentes para 95% varianza: {p.n_components_} de 64 originales')
assert p.n_components_ < 64, 'los digitos viven en un manifold de menor dimension'
print('La dimension intrinseca << 64: por eso reducir dimension funciona.')